# Preprocessing et Feature Engineering

In [63]:
import pandas as pd
import numpy as np

## Date conversion and Data loading

In [64]:
df = pd.read_csv("../data/merged_dataset.csv")
df["delivery_time"] = pd.to_datetime(df['delivery_time'])

## Handling Missing Values

Linear interpolation for weather, as it's a continuous time series

In [65]:
df = df.groupby('site_name').apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=False)

Power is proportional to the cube of wind speed:

In [66]:
df['wind_speed_100m_cube'] = df['wind_speed_100m'] ** 3

## Feature Engineering: Wind Physics

Wind Shear: Difference in speed between 10m and 100m:

In [67]:
df['wind_shear'] = df['wind_speed_100m'] - df['wind_speed_10m']

## Feature Engineering: Circular Encoding (Direction & Time)

Wind Direction (avoiding the 359° to 0° jump)

In [68]:
df['wind_dir_100m_sin'] = np.sin(np.radians(df['wind_direction_100m']))
df['wind_dir_100m_cos'] = np.cos(np.radians(df['wind_direction_100m']))

Time-based features (Daily and Seasonal cycles)

In [69]:
df['hour'] = df['delivery_time'].dt.hour
df['month'] = df['delivery_time'].dt.month

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 23)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 23)
df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 11)
df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 11)

## Target Normalization (Critical for multi-site modeling)
Predicting Load Factor (0 to 1) instead of raw MW

In [70]:
# df['target'] = df['production'] / df['installed_capacity']

Remove columns that would cause data leakage or are redundant

In [71]:
cols_to_drop = [ 'hour', 'month', 'wind_direction_10m', 'wind_direction_100m'] # 'production',
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

Ajout de la variable target : utilization_rate


In [72]:
# Ajout de utilization_rate one hot encoding pour le nom des sites et keep site_name 

df['utilization_rate'] = df['production'] / df['installed_capacity']
site_name_keep = df['site_name']
df = pd.get_dummies(df, columns=['site_name'], prefix='site')

#Add site_name_keep au début du DataFrame
df.insert(0, 'site_name', site_name_keep)

# Ensuite supprimer 'level_1' 
df = df.drop(columns=['level_1'])


In [73]:
df.head()

,site_name,delivery_time,production,installed_capacity,wind_speed_10m,wind_speed_100m,wind_gusts_10m,temperature_2m,dewpoint_2m,apparent_temperature,...,site_Belwind Phase 1,site_Mermaid Offshore WP,site_Nobelwind Offshore Windpark,site_Norther Offshore WP,site_Northwester 2,site_Northwind,site_Rentel Offshore WP,site_Seastar Offshore WP,site_Thorntonbank - C-Power - Area NE,site_Thorntonbank - C-Power - Area SW
0,Belwind Phase 1,2023-01-01 00:00:00+00:00,147.7025,171.0,14.603082,19.897738,20.7,12.25,8.85,4.282408,...,True,False,False,False,False,False,False,False,False,False
1,Belwind Phase 1,2023-01-01 01:00:00+00:00,146.1775,171.0,16.182089,21.681328,20.8,12.10,8.80,3.290131,...,True,False,False,False,False,False,False,False,False,False
2,Belwind Phase 1,2023-01-01 02:00:00+00:00,146.1800,171.0,17.969420,23.809662,24.1,11.85,9.50,2.291797,...,True,False,False,False,False,False,False,False,False,False
3,Belwind Phase 1,2023-01-01 03:00:00+00:00,146.5050,171.0,14.792228,19.860010,23.9,11.80,9.85,4.007824,...,True,False,False,False,False,False,False,False,False,False
4,Belwind Phase 1,2023-01-01 04:00:00+00:00,146.6950,171.0,15.001333,19.915070,19.7,11.75,9.30,3.694952,...,True,False,False,False,False,False,False,False,False,False


In [74]:
df.to_csv("../data/processed_dataset.csv")